In [1]:
!pip install transformers accelerate torch -q

In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
from tqdm import tqdm

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

model.to(device)
model.eval()

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [5]:
df = pd.read_csv("/kaggle/input/datasets/trangle2110/ptdlnc-group-5/full_clauses_labeled_final.csv")

print(df.head())
print(df.info())

   Review_ID  clause_id Username ID_Address        Address  Rating Language  \
0          1        1.1  Mashajo         AB  An Bang Beach       5       ru   
1          1        1.2  Mashajo         AB  An Bang Beach       5       ru   
2          1        1.3  Mashajo         AB  An Bang Beach       5       ru   
3          1        1.4  Mashajo         AB  An Bang Beach       5       ru   
4          1        1.5  Mashajo         AB  An Bang Beach       5       ru   

                                              Clause code  
0  It is about 10 to 15 minutes away from the old...   CE  
1  It's pretty quiet and there's no feeling of a ...   SI  
2  There are a lot of cozy cafes and restaurants ...   CE  
3  The place is perfect for relaxing rest: you ca...   CE  
4  And it's particularly beautiful here tonight, ...   CE  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 288126 entries, 0 to 288125
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype  
---  ------  

In [6]:
def get_sentiment_score(text):
    try:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=1)

        probs = probs.cpu().numpy()[0]

        # label order: negative, neutral, positive
        score = (-1 * probs[0]) + (0 * probs[1]) + (1 * probs[2])

        return float(score)

    except:
        return None

In [7]:
tqdm.pandas()

df["senti_score"] = df["Clause"].progress_apply(get_sentiment_score)

100%|██████████| 288126/288126 [41:40<00:00, 115.21it/s]


In [8]:
df

,Review_ID,clause_id,Username,ID_Address,Address,Rating,Language,Clause,code,senti_score
0,1,1.1,Mashajo,AB,An Bang Beach,5,ru,It is about 10 to 15 minutes away from the old...,CE,0.946638
1,1,1.2,Mashajo,AB,An Bang Beach,5,ru,It's pretty quiet and there's no feeling of a ...,SI,0.368487
2,1,1.3,Mashajo,AB,An Bang Beach,5,ru,There are a lot of cozy cafes and restaurants ...,CE,0.781911
3,1,1.4,Mashajo,AB,An Bang Beach,5,ru,The place is perfect for relaxing rest: you ca...,CE,0.971186
4,1,1.5,Mashajo,AB,An Bang Beach,5,ru,"And it's particularly beautiful here tonight, ...",CE,0.987315
...,...,...,...,...,...,...,...,...,...,...
288121,65956,65956.1,Nico Venter,WS,White Sand Dunes,1,en,"Great place for taking pictures,",VA,0.962515
288122,65956,65956.2,Nico Venter,WS,White Sand Dunes,1,en,"be warned they don't give proper information, ...",VA,-0.872530
288123,65957,65957.1,Babydino,WS,White Sand Dunes,1,ko,Watch out for those of you who are going in th...,VA,-0.037172
288124,65957,65957.2,Babydino,WS,White Sand Dunes,1,ko,"It's as if it's helping,",VA,0.723146


In [9]:
df.to_csv("/kaggle/working/Sentiment_Score_LLM.csv", index=False)